# U.S. Visa Sponsorship Analytics Dashboard
## APAN 5450 - Cloud Computing | Group 1
---


### Data Sources
- **H-1B Dataset**: USCIS H-1B Employer Data Hub — FY2009 to FY2026 (1,043,479 rows)
- **PERM Dataset**: DOL Foreign Labor Certification Data — FY2008 to FY2026 (1,652,759 rows)

### AWS Services Used
- **EC2**: t3.small instance — hosts Streamlit dashboard on port 8501

## Dashboard Purpose

This interactive dashboard helps **international job seekers** identify U.S. employers who sponsor H-1B work visas and PERM (green card) applications. It answers five core questions:

1. Which companies sponsor H-1B visas the most?
2. Which companies sponsor PERM (green card) applications?
3. How has sponsorship activity changed over time?
4. Which states have the most active sponsors?
5. What salary can I expect from a sponsored role?

---

## Step 1: Install Required Libraries

Install all dependencies needed to run the Streamlit dashboard on EC2.

In [ ]:
# Install required Python packages on EC2
# Run this cell once during initial EC2 setup
!pip install streamlit pandas plotly psycopg2-binary

## Step 2: Database Connection

Connect to the AWS RDS PostgreSQL instance that stores the cleaned H-1B and PERM data.
Data was loaded via AWS Glue ETL jobs after cleaning and standardization.

In [ ]:
# =============================================================================
# LIBRARY IMPORTS
# =============================================================================
import streamlit as st        # Web dashboard framework
import pandas as pd            # Data manipulation
import plotly.express as px    # Interactive visualizations
import psycopg2                # PostgreSQL database connector

# =============================================================================
# DATABASE CONNECTION CONFIGURATION
# RDS PostgreSQL instance hosted on AWS (us-east-1)
# Data cleaned and loaded via AWS Glue ETL jobs
# =============================================================================
DB_HOST = "visa-group1-postgres.clyiuasiev48.us-east-1.rds.amazonaws.com"
DB_PORT = 5432
DB_NAME = "visa_analytics"
DB_USER = "postgres"
DB_PASS = "postgres"

# Cache the database connection across Streamlit sessions to avoid
# re-connecting on every user interaction (performance optimization)
@st.cache_resource
def get_connection():
    return psycopg2.connect(
        host=DB_HOST, port=DB_PORT,
        dbname=DB_NAME, user=DB_USER, password=DB_PASS
    )

# Cache query results to avoid redundant database calls on re-renders.
# Results are invalidated when query parameters change (e.g. year slider).
@st.cache_data
def run_query(query, params=None):
    conn = get_connection()
    return pd.read_sql(query, conn, params=params)

## Step 3: Page Configuration and Styling

Configure the Streamlit page layout and apply custom CSS for the dark theme dashboard.

In [ ]:
# =============================================================================
# PAGE CONFIGURATION
# Wide layout for multi-column dashboard design
# =============================================================================
st.set_page_config(
    page_title="U.S. Visa Sponsorship Analytics",
    layout="wide",
    initial_sidebar_state="expanded"
)

# =============================================================================
# CUSTOM CSS STYLING
# Dark theme with custom KPI cards and section titles
# Color scheme:
#   #e94560 = red/pink (primary accent, H-1B charts)
#   #2ecc71 = green (approvals, PERM charts)
#   #00b4d8 = blue (wage charts)
#   #a8b2d8 = muted blue (secondary text)
#   #0f1117 = dark background
#   #16213e = card background
# =============================================================================
st.markdown("""
<style>
[data-testid="stAppViewContainer"] { background-color: #0f1117; }
[data-testid="stSidebar"] { background-color: #1a1a2e; }
.kpi-card {
    background: #16213e;
    border: 1px solid #0f3460;
    border-radius: 12px;
    padding: 1.2rem;
    text-align: center;
    margin-bottom: 1rem;
}
.kpi-value { color: #e94560; font-size: 1.6rem; font-weight: 700; }
.kpi-label { color: #a8b2d8; font-size: 0.8rem; margin-top: 0.3rem; }
.kpi-sublabel { color: #2ecc71; font-size: 0.75rem; margin-top: 0.2rem; }
.section-title {
    color: #e94560;
    font-size: 1.1rem;
    font-weight: 600;
    border-left: 3px solid #e94560;
    padding-left: 0.7rem;
    margin: 1rem 0;
}
</style>
""", unsafe_allow_html=True)

# Dashboard header
st.markdown("<h1 style='color:#e94560;text-align:center;padding-top:1rem;'>U.S. Visa Sponsorship Analytics</h1>", unsafe_allow_html=True)
st.markdown("<p style='color:#a8b2d8;text-align:center;'>Explore H-1B and PERM sponsorship trends across employers, states, and years | APAN 5450 Group 1</p>", unsafe_allow_html=True)
st.markdown("---")

## Step 4: Sidebar Filters

A global year range slider in the sidebar controls all queries across all tabs.
This allows users to filter data to specific time periods of interest.

In [ ]:
# =============================================================================
# SIDEBAR - GLOBAL FILTERS
# Year range slider applies to all tabs via parameterized SQL queries.
# Default range: 2015-2026 (most relevant years for current job market)
# =============================================================================
st.sidebar.markdown("<h2 style='color:#e94560;'>Filters</h2>", unsafe_allow_html=True)
st.sidebar.markdown("---")
year_range = st.sidebar.slider("Year Range", min_value=2008, max_value=2026, value=(2015, 2026))
st.sidebar.markdown("---")
st.sidebar.markdown(
    "<div style='background:#16213e;border-radius:8px;padding:0.8rem;'>"
    "<p style='color:#e94560;font-weight:600;margin:0;'>About</p>"
    "<p style='color:#a8b2d8;font-size:0.75rem;margin:0.3rem 0 0 0;'>"
    "This dashboard helps job seekers identify U.S. employers who sponsor "
    "H-1B and PERM visas. Data sourced from USCIS and DOL public records."
    "</p></div>",
    unsafe_allow_html=True
)
st.sidebar.markdown("<br><small style='color:#a8b2d8;'>APAN 5450 | Group 1<br>Columbia University SPS</small>", unsafe_allow_html=True)

## Step 5: KPI Summary Cards

Five top-level metrics provide an at-a-glance summary of the visa sponsorship landscape.
All metrics respond dynamically to the year range filter in the sidebar.

In [ ]:
# =============================================================================
# KPI SUMMARY CARDS
# Five top-level metrics computed dynamically from RDS based on selected year range.
# Metrics:
#   1. Total H-1B Approvals     — SUM of new_employment_approval
#   2. H-1B Approval Rate       — approvals / (approvals + denials)
#   3. PERM Certifications      — COUNT of certified PERM cases
#   4. Top H-1B Sponsor         — employer with most approvals
#   5. Avg PERM Wage Offer      — AVG of wage_offer_from (min wage)
# =============================================================================
with st.spinner("Loading key metrics..."):
    try:
        # Total H-1B new employment approvals in selected year range
        total_h1b = run_query(
            "SELECT SUM(new_employment_approval) as total FROM h1b_sponsors "
            "WHERE fiscal_year BETWEEN %s AND %s",
            params=(year_range[0], year_range[1])
        )["total"][0]

        # Total H-1B new employment denials in selected year range
        total_denial = run_query(
            "SELECT SUM(new_employment_denial) as total FROM h1b_sponsors "
            "WHERE fiscal_year BETWEEN %s AND %s",
            params=(year_range[0], year_range[1])
        )["total"][0]

        # Approval rate = approvals / (approvals + denials)
        # Guard against division by zero using conditional check
        approval_rate = round(
            (total_h1b / (total_h1b + total_denial)) * 100, 1
        ) if (total_h1b + total_denial) > 0 else 0

        # Employer with highest total new employment approvals
        top_employer = run_query(
            "SELECT employer_name FROM h1b_sponsors WHERE fiscal_year BETWEEN %s AND %s "
            "GROUP BY employer_name ORDER BY SUM(new_employment_approval) DESC LIMIT 1",
            params=(year_range[0], year_range[1])
        )["employer_name"][0]

        # Total certified PERM applications (green card pathway)
        total_perm = run_query(
            "SELECT COUNT(*) as total FROM perm_records "
            "WHERE filing_year BETWEEN %s AND %s AND case_status = 'Certified'",
            params=(year_range[0], year_range[1])
        )["total"][0]

        # Average minimum wage offer from certified PERM records
        avg_wage = run_query(
            "SELECT AVG(wage_offer_from) as avg FROM perm_records "
            "WHERE filing_year BETWEEN %s AND %s AND wage_offer_from > 0",
            params=(year_range[0], year_range[1])
        )["avg"][0]

        # Render KPI cards in 5 equal columns
        col1, col2, col3, col4, col5 = st.columns(5)
        with col1:
            st.markdown(f'<div class="kpi-card"><div class="kpi-value">{int(total_h1b):,}</div><div class="kpi-label">Total H-1B Approvals</div></div>', unsafe_allow_html=True)
        with col2:
            st.markdown(f'<div class="kpi-card"><div class="kpi-value">{approval_rate}%</div><div class="kpi-label">H-1B Approval Rate</div><div class="kpi-sublabel">approvals / total petitions</div></div>', unsafe_allow_html=True)
        with col3:
            st.markdown(f'<div class="kpi-card"><div class="kpi-value">{int(total_perm):,}</div><div class="kpi-label">PERM Certifications</div></div>', unsafe_allow_html=True)
        with col4:
            short_name = top_employer[:14] + ".." if len(top_employer) > 14 else top_employer
            st.markdown(f'<div class="kpi-card"><div class="kpi-value" style="font-size:1rem;">{short_name}</div><div class="kpi-label">Top H-1B Sponsor</div></div>', unsafe_allow_html=True)
        with col5:
            avg_wage_fmt = f"${int(avg_wage):,}" if avg_wage else "N/A"
            st.markdown(f'<div class="kpi-card"><div class="kpi-value">{avg_wage_fmt}</div><div class="kpi-label">Avg PERM Wage Offer</div></div>', unsafe_allow_html=True)
    except Exception as e:
        st.warning(f"KPI cards could not load: {e}")

st.markdown("<br>", unsafe_allow_html=True)

## Step 6: Main Dashboard Tabs

The dashboard is organized into 5 tabs, each answering a core user question:

| Tab | Question Answered |
|-----|-------------------|
| Top Sponsors | Which companies sponsor the most? |
| Trends Over Time | How has sponsorship changed year over year? |
| By State | Where are the most sponsors located? |
| Wage Analysis | What salary can I expect from a sponsored role? |
| Search Employer | Does my target company sponsor visas? |

In [ ]:
# Initialize all 5 tabs
tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "Top Sponsors",
    "Trends Over Time",
    "By State",
    "Wage Analysis",
    "Search Employer"
])

## Step 7: Tab 1 — Top Sponsors

Shows top 15 H-1B and PERM employers by volume, plus an approvals vs denials comparison.
Users can identify high-volume sponsors and assess their reliability (high approval rate = safer bet).

In [ ]:
# =============================================================================
# TAB 1 - TOP SPONSORS
# Three charts:
#   1. Top 15 H-1B sponsors by new employment approvals (horizontal bar)
#   2. Top 15 PERM employers by certified cases (horizontal bar)
#   3. Approvals vs Denials comparison for top 10 H-1B employers (grouped bar)
# =============================================================================
with tab1:
    col_l, col_r = st.columns(2)
    with col_l:
        st.markdown('<div class="section-title">Top H-1B Sponsors</div>', unsafe_allow_html=True)
        with st.spinner("Loading..."):
            try:
                # Aggregate new employment approvals and denials per employer
                # Approval rate calculated using NULLIF to avoid division by zero
                df = run_query(
                    "SELECT employer_name, "
                    "SUM(new_employment_approval) AS approvals, "
                    "SUM(new_employment_denial) AS denials, "
                    "ROUND(SUM(new_employment_approval)::numeric / "
                    "NULLIF(SUM(new_employment_approval) + SUM(new_employment_denial), 0) * 100, 1) AS approval_rate "
                    "FROM h1b_sponsors WHERE fiscal_year BETWEEN %s AND %s "
                    "GROUP BY employer_name ORDER BY approvals DESC LIMIT 15",
                    params=(year_range[0], year_range[1])
                )
                fig = px.bar(
                    df, x="approvals", y="employer_name", orientation="h",
                    color="approvals", color_continuous_scale="reds",
                    hover_data={"denials": True, "approval_rate": True},
                    labels={"approvals": "Approvals", "employer_name": "",
                            "denials": "Denials", "approval_rate": "Approval Rate %"}
                )
                fig.update_layout(
                    plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)",
                    font_color="white", coloraxis_showscale=False,
                    yaxis=dict(autorange="reversed"), margin=dict(l=0,r=0,t=10,b=0)
                )
                st.plotly_chart(fig, use_container_width=True)
            except Exception as e:
                st.warning(f"Could not load: {e}")

    with col_r:
        st.markdown('<div class="section-title">Top PERM Employers</div>', unsafe_allow_html=True)
        with st.spinner("Loading..."):
            try:
                # Count certified PERM applications per employer
                # PERM = Permanent Labor Certification (DOL green card pathway)
                df = run_query(
                    "SELECT employer_name, COUNT(*) AS total FROM perm_records "
                    "WHERE filing_year BETWEEN %s AND %s AND case_status = 'Certified' "
                    "GROUP BY employer_name ORDER BY total DESC LIMIT 15",
                    params=(year_range[0], year_range[1])
                )
                fig = px.bar(
                    df, x="total", y="employer_name", orientation="h",
                    color="total", color_continuous_scale="greens",
                    labels={"total": "Certified Cases", "employer_name": ""}
                )
                fig.update_layout(
                    plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)",
                    font_color="white", coloraxis_showscale=False,
                    yaxis=dict(autorange="reversed"), margin=dict(l=0,r=0,t=10,b=0)
                )
                st.plotly_chart(fig, use_container_width=True)
            except Exception as e:
                st.warning(f"Could not load: {e}")

    # Grouped bar chart: approvals vs denials for top 10 H-1B employers
    # Melt transforms wide format (approvals, denials columns) to long format for grouped bar
    st.markdown('<div class="section-title">H-1B Approvals vs Denials - Top 10 Employers</div>', unsafe_allow_html=True)
    with st.spinner("Loading..."):
        try:
            df = run_query(
                "SELECT employer_name, SUM(new_employment_approval) AS approvals, "
                "SUM(new_employment_denial) AS denials FROM h1b_sponsors "
                "WHERE fiscal_year BETWEEN %s AND %s "
                "GROUP BY employer_name ORDER BY approvals DESC LIMIT 10",
                params=(year_range[0], year_range[1])
            )
            # Melt dataframe from wide to long format for grouped bar chart
            df_melted = df.melt(
                id_vars="employer_name",
                value_vars=["approvals", "denials"],
                var_name="type", value_name="count"
            )
            fig = px.bar(
                df_melted, x="employer_name", y="count", color="type", barmode="group",
                color_discrete_map={"approvals": "#2ecc71", "denials": "#e94560"},
                labels={"employer_name": "", "count": "Count", "type": ""}
            )
            fig.update_layout(
                plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)",
                font_color="white", xaxis_tickangle=-30,
                legend=dict(bgcolor="rgba(0,0,0,0)")
            )
            st.plotly_chart(fig, use_container_width=True)
        except Exception as e:
            st.warning(f"Could not load: {e}")

## Step 8: Tab 2 — Trends Over Time

Area charts showing H-1B and PERM volume trends from 2015 onwards,
plus industry breakdowns using NAICS codes from both datasets.
Data starts from 2015 to focus on the most relevant period for job seekers.

In [ ]:
# =============================================================================
# TAB 2 - TRENDS OVER TIME
# Four charts:
#   1. H-1B approvals area chart (2015 onwards)
#   2. PERM certifications area chart (2015 onwards, same format)
#   3. Top H-1B industries by approvals (industry_naics_code column)
#   4. Top PERM industries by certifications (naics_title column)
# Note: Both area charts start from 2015 regardless of sidebar year filter
#       for consistent trend visualization.
# =============================================================================
with tab2:
    # H-1B approval trend — area chart with fill for visual emphasis
    st.markdown('<div class="section-title">H-1B Approvals Over Time</div>', unsafe_allow_html=True)
    with st.spinner("Loading..."):
        try:
            df = run_query(
                "SELECT fiscal_year AS year, SUM(new_employment_approval) AS approvals "
                "FROM h1b_sponsors WHERE fiscal_year >= 2015 "
                "GROUP BY fiscal_year ORDER BY fiscal_year"
            )
            fig = px.area(
                df, x="year", y="approvals",
                color_discrete_sequence=["#e94560"],
                labels={"year": "Year", "approvals": "Total Approvals"}
            )
            fig.update_traces(fill="tozeroy", fillcolor="rgba(233,69,96,0.2)")
            fig.update_layout(plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)", font_color="white")
            st.plotly_chart(fig, use_container_width=True)
        except Exception as e:
            st.warning(f"Could not load: {e}")

    # PERM certification trend — identical area chart format for visual consistency
    st.markdown('<div class="section-title">PERM Certifications Over Time</div>', unsafe_allow_html=True)
    with st.spinner("Loading..."):
        try:
            df = run_query(
                "SELECT filing_year AS year, COUNT(*) AS approvals FROM perm_records "
                "WHERE case_status = 'Certified' AND filing_year >= 2015 "
                "GROUP BY filing_year ORDER BY filing_year"
            )
            fig = px.area(
                df, x="year", y="approvals",
                color_discrete_sequence=["#00b4d8"],
                labels={"year": "Year", "approvals": "Total Certifications"}
            )
            fig.update_traces(fill="tozeroy", fillcolor="rgba(0,180,216,0.2)")
            fig.update_layout(plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)", font_color="white")
            st.plotly_chart(fig, use_container_width=True)
        except Exception as e:
            st.warning(f"Could not load: {e}")

    col_l, col_r = st.columns(2)
    with col_l:
        # H-1B industry breakdown using industry_naics_code column
        # Values are pre-labeled by Glue ETL (e.g. "54 - Professional, Scientific...")
        st.markdown('<div class="section-title">Top H-1B Approvals by Industry</div>', unsafe_allow_html=True)
        with st.spinner("Loading..."):
            try:
                df = run_query("""
                    SELECT industry_naics_code AS industry,
                           SUM(new_employment_approval) AS approvals
                    FROM h1b_sponsors
                    WHERE fiscal_year BETWEEN %s AND %s
                    AND industry_naics_code IS NOT NULL
                    AND industry_naics_code != ''
                    GROUP BY industry_naics_code
                    ORDER BY approvals DESC
                    LIMIT 15
                """, params=(year_range[0], year_range[1]))
                fig = px.bar(
                    df, x="approvals", y="industry", orientation="h",
                    color="approvals", color_continuous_scale="reds",
                    labels={"approvals": "Total Approvals", "industry": ""}
                )
                fig.update_layout(
                    plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)",
                    font_color="white", coloraxis_showscale=False,
                    yaxis=dict(autorange="reversed"), margin=dict(l=0,r=0,t=10,b=0)
                )
                st.plotly_chart(fig, use_container_width=True)
            except Exception as e:
                st.warning(f"Could not load: {e}")

    with col_r:
        # PERM industry breakdown using naics_title column
        # naics_title provides human-readable names (e.g. "Custom Computer Programming Services")
        st.markdown('<div class="section-title">Top PERM Certifications by Industry (NAICS)</div>', unsafe_allow_html=True)
        with st.spinner("Loading..."):
            try:
                df = run_query("""
                    SELECT naics_title AS industry,
                           COUNT(*) AS certifications
                    FROM perm_records
                    WHERE filing_year BETWEEN %s AND %s
                    AND case_status = 'Certified'
                    AND naics_title IS NOT NULL
                    AND naics_title != ''
                    GROUP BY naics_title
                    ORDER BY certifications DESC
                    LIMIT 15
                """, params=(year_range[0], year_range[1]))
                fig = px.bar(
                    df, x="certifications", y="industry", orientation="h",
                    color="certifications", color_continuous_scale="greens",
                    labels={"certifications": "Total Certifications", "industry": ""}
                )
                fig.update_layout(
                    plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)",
                    font_color="white", coloraxis_showscale=False,
                    yaxis=dict(autorange="reversed"), margin=dict(l=0,r=0,t=10,b=0)
                )
                st.plotly_chart(fig, use_container_width=True)
            except Exception as e:
                st.warning(f"Could not load: {e}")

## Step 9: Tab 3 — By State

U.S. choropleth maps and bar charts for both H-1B and PERM by state.
Helps users decide where to focus their job search geographically.

> **Note:** `employer_state` is already normalized to 2-letter codes by the Glue ETL job.
> NULL states are excluded from all state-level visualizations.

In [ ]:
# =============================================================================
# TAB 3 - BY STATE
# Four charts arranged in 2x2 grid:
#   Top row:    H-1B choropleth map | PERM choropleth map
#   Bottom row: Top 10 H-1B states bar | Top 10 PERM states bar
# Note: employer_state already normalized to 2-letter codes by Glue ETL.
#       NULL states excluded to prevent choropleth rendering errors.
# =============================================================================
with tab3:
    col_l, col_r = st.columns(2)
    with col_l:
        # H-1B choropleth map — color intensity represents total approvals per state
        st.markdown('<div class="section-title">H-1B Approvals by State</div>', unsafe_allow_html=True)
        with st.spinner("Loading map..."):
            try:
                df = run_query(
                    "SELECT employer_state AS state, SUM(new_employment_approval) AS total "
                    "FROM h1b_sponsors WHERE fiscal_year BETWEEN %s AND %s "
                    "AND employer_state IS NOT NULL "
                    "GROUP BY employer_state ORDER BY total DESC",
                    params=(year_range[0], year_range[1])
                )
                fig = px.choropleth(
                    df, locations="state", locationmode="USA-states",
                    color="total", scope="usa", color_continuous_scale="reds",
                    labels={"total": "Total Approvals", "state": "State"}
                )
                fig.update_layout(
                    plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)",
                    font_color="white", geo=dict(bgcolor="rgba(0,0,0,0)")
                )
                st.plotly_chart(fig, use_container_width=True)
            except Exception as e:
                st.warning(f"Could not load: {e}")

    with col_r:
        # PERM choropleth map — color intensity represents total certified cases per state
        st.markdown('<div class="section-title">PERM Certifications by State</div>', unsafe_allow_html=True)
        with st.spinner("Loading map..."):
            try:
                df = run_query(
                    "SELECT employer_state AS state, COUNT(*) AS total FROM perm_records "
                    "WHERE filing_year BETWEEN %s AND %s AND case_status = 'Certified' "
                    "AND employer_state IS NOT NULL "
                    "GROUP BY employer_state ORDER BY total DESC",
                    params=(year_range[0], year_range[1])
                )
                fig = px.choropleth(
                    df, locations="state", locationmode="USA-states",
                    color="total", scope="usa", color_continuous_scale="greens",
                    labels={"total": "Total Certifications", "state": "State"}
                )
                fig.update_layout(
                    plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)",
                    font_color="white", geo=dict(bgcolor="rgba(0,0,0,0)")
                )
                st.plotly_chart(fig, use_container_width=True)
            except Exception as e:
                st.warning(f"Could not load: {e}")

    # Bar charts for precise state ranking
    col_l, col_r = st.columns(2)
    with col_l:
        st.markdown('<div class="section-title">Top 10 States - H-1B Approvals</div>', unsafe_allow_html=True)
        with st.spinner("Loading..."):
            try:
                df = run_query(
                    "SELECT employer_state AS state, SUM(new_employment_approval) AS total "
                    "FROM h1b_sponsors WHERE fiscal_year BETWEEN %s AND %s "
                    "AND employer_state IS NOT NULL "
                    "GROUP BY employer_state ORDER BY total DESC LIMIT 10",
                    params=(year_range[0], year_range[1])
                )
                fig = px.bar(df, x="state", y="total", color="total", color_continuous_scale="reds",
                    labels={"state": "State", "total": "Total Approvals"})
                fig.update_layout(plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)",
                    font_color="white", coloraxis_showscale=False)
                st.plotly_chart(fig, use_container_width=True)
            except Exception as e:
                st.warning(f"Could not load: {e}")

    with col_r:
        st.markdown('<div class="section-title">Top 10 States - PERM Certifications</div>', unsafe_allow_html=True)
        with st.spinner("Loading..."):
            try:
                df = run_query(
                    "SELECT employer_state AS state, COUNT(*) AS total FROM perm_records "
                    "WHERE filing_year BETWEEN %s AND %s AND case_status = 'Certified' "
                    "AND employer_state IS NOT NULL "
                    "GROUP BY employer_state ORDER BY total DESC LIMIT 10",
                    params=(year_range[0], year_range[1])
                )
                fig = px.bar(df, x="state", y="total", color="total", color_continuous_scale="greens",
                    labels={"state": "State", "total": "Total Certifications"})
                fig.update_layout(plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)",
                    font_color="white", coloraxis_showscale=False)
                st.plotly_chart(fig, use_container_width=True)
            except Exception as e:
                st.warning(f"Could not load: {e}")

## Step 10: Tab 4 — Wage Analysis

Helps users understand salary expectations for sponsored roles using PERM wage data.

> **Data Quality Note:** `wage_offer_from > 20000` filter is applied to remove anomalies caused by
> ETL schema mismatches in earlier PERM dataset files (particularly the 2017 dip).
> `HAVING COUNT(*) > 50/100` ensures statistical significance for averages.

In [ ]:
# =============================================================================
# TAB 4 - WAGE ANALYSIS
# Three charts:
#   1. Top 15 job titles by average minimum wage (PERM wage_offer_from)
#   2. Average wage by NAICS industry (naics_title)
#   3. Wage range trend over time (avg min and max wage per year)
# Data quality filters:
#   - wage_offer_from > 20000: removes anomalies from ETL schema mismatches
#   - HAVING COUNT(*) > 50/100: ensures statistical significance
# =============================================================================
with tab4:
    col_l, col_r = st.columns(2)
    with col_l:
        # Top 15 job titles by average minimum wage offer
        # HAVING COUNT(*) > 50 ensures statistical significance (excludes rare titles)
        st.markdown('<div class="section-title">Top 15 Job Titles by Average Wage</div>', unsafe_allow_html=True)
        with st.spinner("Loading..."):
            try:
                df = run_query(
                    "SELECT job_title, ROUND(AVG(wage_offer_from)::numeric, 0) AS avg_wage, "
                    "COUNT(*) AS total_cases FROM perm_records "
                    "WHERE filing_year BETWEEN %s AND %s "
                    "AND wage_offer_from > 0 AND job_title IS NOT NULL "
                    "GROUP BY job_title HAVING COUNT(*) > 50 "
                    "ORDER BY avg_wage DESC LIMIT 15",
                    params=(year_range[0], year_range[1])
                )
                fig = px.bar(
                    df, x="avg_wage", y="job_title", orientation="h",
                    color="avg_wage", color_continuous_scale="blues",
                    hover_data={"total_cases": True},
                    labels={"avg_wage": "Avg Wage ($)", "job_title": "", "total_cases": "Total Cases"}
                )
                fig.update_layout(
                    plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)",
                    font_color="white", coloraxis_showscale=False,
                    yaxis=dict(autorange="reversed"), margin=dict(l=0,r=0,t=10,b=0)
                )
                st.plotly_chart(fig, use_container_width=True)
            except Exception as e:
                st.warning(f"Could not load: {e}")

    with col_r:
        # Average wage grouped by NAICS industry title
        # wage_offer_from > 20000 filters out data anomalies from schema mismatches
        # HAVING COUNT(*) > 100 ensures industry has sufficient sample size
        st.markdown('<div class="section-title">Average Wage by Industry (NAICS)</div>', unsafe_allow_html=True)
        with st.spinner("Loading..."):
            try:
                df = run_query(
                    "SELECT naics_title AS industry, "
                    "ROUND(AVG(wage_offer_from)::numeric, 0) AS avg_wage, "
                    "COUNT(*) AS total_cases FROM perm_records "
                    "WHERE filing_year BETWEEN %s AND %s "
                    "AND wage_offer_from > 20000 "
                    "AND naics_title IS NOT NULL AND naics_title != '' "
                    "GROUP BY naics_title HAVING COUNT(*) > 100 "
                    "ORDER BY avg_wage DESC LIMIT 15",
                    params=(year_range[0], year_range[1])
                )
                fig = px.bar(
                    df, x="avg_wage", y="industry", orientation="h",
                    color="avg_wage", color_continuous_scale="purples",
                    hover_data={"total_cases": True},
                    labels={"avg_wage": "Avg Wage ($)", "industry": "", "total_cases": "Total Cases"}
                )
                fig.update_layout(
                    plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)",
                    font_color="white", coloraxis_showscale=False,
                    yaxis=dict(autorange="reversed"), margin=dict(l=0,r=0,t=10,b=0)
                )
                st.plotly_chart(fig, use_container_width=True)
            except Exception as e:
                st.warning(f"Could not load: {e}")

    # Wage range trend over time — dual line chart shows min and max wage offer per year
    # Helps users understand wage growth trajectory over time
    st.markdown('<div class="section-title">Wage Offer Range Over Time</div>', unsafe_allow_html=True)
    with st.spinner("Loading..."):
        try:
            df = run_query(
                "SELECT filing_year AS year, "
                "ROUND(AVG(wage_offer_from)::numeric, 0) AS avg_min, "
                "ROUND(AVG(wage_offer_to)::numeric, 0) AS avg_max "
                "FROM perm_records "
                "WHERE wage_offer_from > 20000 AND wage_offer_to > 20000 "
                "GROUP BY filing_year ORDER BY filing_year"
            )
            fig = px.line(
                df, x="year", y=["avg_min", "avg_max"], markers=True,
                color_discrete_map={"avg_min": "#00b4d8", "avg_max": "#e94560"},
                labels={"year": "Year", "value": "Wage ($)", "variable": "Type"}
            )
            fig.update_layout(
                plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)",
                font_color="white", legend=dict(bgcolor="rgba(0,0,0,0)")
            )
            st.plotly_chart(fig, use_container_width=True)
        except Exception as e:
            st.warning(f"Could not load: {e}")

## Step 11: Tab 5 — Search Employer

Allows users to search for a specific employer by name and view their complete
H-1B and PERM sponsorship history with approval rates.

**Key design decisions:**
- Search term is **uppercased** to match Glue-cleaned employer names (all UPPERCASE in DB)
- Grouped by `employer_name + employer_state` to handle subsidiaries in different locations
- `HAVING` clause filters out zero-activity records (rows with 0 approvals and 0 denials)
- Chart aggregates by year combining all subsidiaries/states for clean trend visualization

In [ ]:
# =============================================================================
# TAB 5 - SEARCH EMPLOYER
# Interactive employer search showing H-1B and PERM history side by side.
# Key design decisions:
#   - search.upper().strip(): matches Glue-cleaned UPPERCASE employer names
#   - GROUP BY employer_name + employer_state: handles subsidiaries by location
#   - HAVING clause: removes zero-activity rows from results
#   - chart_df aggregates by year: combines all subsidiaries for trend view
# =============================================================================
with tab5:
    st.markdown('<div class="section-title">Search by Employer</div>', unsafe_allow_html=True)
    st.markdown(
        "<p style='color:#a8b2d8;font-size:0.85rem;'>Search for a specific employer to see "
        "their H-1B and PERM sponsorship history, approval rates, and wage offers.</p>",
        unsafe_allow_html=True
    )
    search = st.text_input(
        "Enter employer name",
        placeholder="e.g. Google, Amazon, Microsoft, Deloitte...",
        key="employer_search_tab"
    )

    if search:
        # Uppercase search term to match Glue ETL cleaned employer names (all UPPERCASE)
        search_term = f"%{search.upper().strip()}%"
        col_l, col_r = st.columns(2)

        with col_l:
            st.markdown("**H-1B Records**")
            with st.spinner("Searching..."):
                try:
                    # Group by year + employer_name + state for subsidiary-level breakdown
                    # HAVING ensures only rows with actual activity are returned
                    # approval_rate_pct uses NULLIF to prevent division by zero
                    df = run_query("""
                        SELECT fiscal_year AS year,
                               employer_state AS state,
                               SUM(new_employment_approval) AS approvals,
                               SUM(new_employment_denial) AS denials,
                               ROUND(SUM(new_employment_approval)::numeric /
                                     NULLIF(SUM(new_employment_approval) + SUM(new_employment_denial), 0) * 100, 1) AS approval_rate_pct
                        FROM h1b_sponsors
                        WHERE employer_name LIKE %s
                        AND fiscal_year BETWEEN %s AND %s
                        GROUP BY fiscal_year, employer_name, employer_state
                        HAVING SUM(new_employment_approval) + SUM(new_employment_denial) > 0
                        ORDER BY year DESC, approvals DESC
                    """, params=(search_term, year_range[0], year_range[1]))
                    if not df.empty:
                        total_approvals = int(df["approvals"].sum())
                        st.success(f"Total H-1B Approvals: {total_approvals:,}")
                        st.dataframe(df, use_container_width=True, hide_index=True)
                        # Aggregate by year for chart (combines all subsidiaries/states)
                        chart_df = df.groupby("year")[["approvals", "denials"]].sum().reset_index()
                        fig = px.bar(
                            chart_df, x="year", y=["approvals", "denials"], barmode="group",
                            color_discrete_map={"approvals": "#2ecc71", "denials": "#e94560"},
                            title=f"H-1B History for '{search}'",
                            labels={"year": "Year", "value": "Count", "variable": ""}
                        )
                        fig.update_layout(
                            plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)",
                            font_color="white", legend=dict(bgcolor="rgba(0,0,0,0)")
                        )
                        st.plotly_chart(fig, use_container_width=True)
                    else:
                        st.info("No H-1B records found.")
                except Exception as e:
                    st.warning(f"Could not load: {e}")

        with col_r:
            st.markdown("**PERM Records**")
            with st.spinner("Searching..."):
                try:
                    # PERM uses case_status to split certified vs denied counts
                    # approval_rate_pct = certified / total applications
                    df = run_query("""
                        SELECT filing_year AS year,
                               employer_state AS state,
                               COUNT(CASE WHEN case_status = 'Certified' THEN 1 END) AS certified,
                               COUNT(CASE WHEN case_status != 'Certified' THEN 1 END) AS denied,
                               ROUND(COUNT(CASE WHEN case_status = 'Certified' THEN 1 END)::numeric /
                                     NULLIF(COUNT(*), 0) * 100, 1) AS approval_rate_pct
                        FROM perm_records
                        WHERE employer_name LIKE %s
                        AND filing_year BETWEEN %s AND %s
                        GROUP BY filing_year, employer_name, employer_state
                        HAVING COUNT(*) > 0
                        ORDER BY year DESC, certified DESC
                    """, params=(search_term, year_range[0], year_range[1]))
                    if not df.empty:
                        total_certified = int(df["certified"].sum())
                        st.success(f"Total PERM Certifications: {total_certified:,}")
                        st.dataframe(df, use_container_width=True, hide_index=True)
                        # Aggregate by year for chart
                        chart_df = df.groupby("year")[["certified", "denied"]].sum().reset_index()
                        fig = px.bar(
                            chart_df, x="year", y=["certified", "denied"], barmode="group",
                            color_discrete_map={"certified": "#2ecc71", "denied": "#e94560"},
                            title=f"PERM History for '{search}'",
                            labels={"year": "Year", "value": "Count", "variable": ""}
                        )
                        fig.update_layout(
                            plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)",
                            font_color="white", legend=dict(bgcolor="rgba(0,0,0,0)")
                        )
                        st.plotly_chart(fig, use_container_width=True)
                    else:
                        st.info("No PERM records found.")
                except Exception as e:
                    st.warning(f"Could not load: {e}")
    else:
        # Default state before search — prompt with example company names
        st.info("Type an employer name above to search. Try: Google, Amazon, Microsoft, Deloitte, Infosys")

## Step 12: Run the Dashboard

Save the code above as `app.py` and run the following command on EC2 to launch the dashboard.

In [ ]:
# Run this command in the EC2 terminal to launch the Streamlit dashboard
# The dashboard will be accessible at http://<EC2_PUBLIC_IP>:8501

# Standard run command:
# streamlit run app.py --server.port 8501 --server.address 0.0.0.0

# To keep it running after terminal closes (background process):
# nohup streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &

print("Dashboard deployment commands:")
print("Standard:    streamlit run app.py --server.port 8501 --server.address 0.0.0.0")
print("Background:  nohup streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &")

---

## Summary

This notebook documents the complete Streamlit dashboard code for the APAN 5450 Group 1 project.

### Data Pipeline Summary
| Stage | Service | Details |
|-------|---------|--------|
| Storage | AWS S3 | Raw H-1B and PERM CSV files |
| ETL | AWS Glue | State normalization, employer name cleaning, schema unification |
| Database | AWS RDS PostgreSQL | `h1b_sponsors` and `perm_records` tables |
| Application | AWS EC2 + Streamlit | Interactive dashboard on port 8501 |
| Monitoring | CloudWatch + EventBridge | Billing alarms, scheduled automation |
| Load Balancer | AWS ALB | Traffic routing |

### Key Technical Decisions
1. **`@st.cache_resource`** — Caches DB connection across sessions (avoids reconnection overhead)
2. **`@st.cache_data`** — Caches query results until parameters change (reduces RDS load)
3. **`wage_offer_from > 20000`** — Removes wage anomalies from ETL schema mismatches in 2017
4. **`search.upper().strip()`** — Matches Glue-cleaned UPPERCASE employer names in database
5. **`NULLIF(..., 0)`** — Prevents division-by-zero in approval rate calculations
6. **No data cleaning in Streamlit** — All cleaning delegated to AWS Glue ETL (separation of concerns)

### Data Sources
- **H-1B**: USCIS H-1B Employer Data Hub — https://www.uscis.gov/tools/reports-and-studies/h-1b-employer-data-hub
- **PERM**: DOL Foreign Labor Certification Data — https://www.dol.gov/agencies/eta/foreign-labor/performance